# ST-GAE: Real-Time Spatiotemporal Anomaly Detection for Toronto Mobility

This notebook implements a **weakly supervised Spatiotemporal Graph Autoencoder (ST-GAE)** for continuous congestion / collision risk scoring on Toronto Bluetooth monitored corridors.

## Approach

Direct collision prediction is unreliable: crashes are rare, sparse, and poorly balanced as labels. Instead we:

1. Learn the **physics of healthy traffic** under ordinary weather and calendar conditions.
2. Score every corridor × 5-minute step by **reconstruction error** $S_{i,t} = \|X_{i,t} - \hat{X}_{i,t}\|^2$.
3. Treat documented collisions as an **evaluation resource** (thresholds, lead-time, PR-AUC), not as scarce training targets.


## Steps

| Step | Section | Purpose |
|---|---|---|
| 1 | Setup, load, clean, normative mask | Reproducible baseline data for training |
| 2 | Feature tensor + corridor graph | Build $X \in \mathbb{R}^{T \times N \times D}$ and adjacency $A$ |
| 3 | ST-GAE train / persist | GCN + GRU autoencoder on normal windows only |
| 4 | Realtime-style scoring | Streaming windows, route thresholds, collision validation |

**Data prerequisite:** upload `route_time_panel_v1.parquet` to Google Drive at `MyDrive/Dissertation/` (built locally with `python -m src.processing --step panel_v1`).

## 0. Colab dependency install

Install the Data Processing + ML stack used by later cells (`pandas`, `numpy`, `sklearn`, `pyarrow`, `matplotlib` `torch`, `torch_geometric`, plus parquet / sklearn helpers).


In [ ]:
# 1. Install standard data science packages and core PyTorch silently
!pip install pandas numpy scikit-learn pyarrow matplotlib torch -q

# 2. Install PyTorch Geometric
!pip install torch-geometric -q

import torch
import torch_geometric

print(
    "Dependencies installed |",
    f"torch={torch.__version__} |",
    f"torch_geometric={torch_geometric.__version__} |",
    f"cuda={torch.cuda.is_available()}",
)

## 1. Environment and configuration

Set up the shared runtime for the rest of the notebook:

- Imports (`numpy`, `pandas`, `pathlib`, etc.) used by cleaning and later modelling cells.
- A single `STGAEConfig` dataclass that holds **all** parameters in one place: Drive paths, train/eval years, rolling window length, model size, learning rate, and normative-mask thresholds.
- Fixed random seeds so tensor construction / training runs are more reproducible across Colab sessions.

**Google Drive layout used**

| Path | Role |
|---|---|
| `/content/drive/MyDrive/Dissertation/` | Project root on Drive |
| `.../route_time_panel_v1.parquet` | Joined analysis panel (input) |
| `.../artifacts/st_gae/` | Saved cleaning QA, scaler, model checkpoints (output) |


**Key config fields (defaults)**

| Field | Default | Meaning |
|---|---|---|
| `train_years` | `(2016,)` | Years used to learn healthy flow |
| `eval_years` | `(2017,)` | Held-out year for anomaly scoring / validation |
| `window_size` | `12` | 12 × 5 min = **60 min** temporal context for the GRU |
| `delay_percentile` | `0.85` | Per-route delay cutoff for the normative mask |
| `precip_threshold_mm` | `2.5` | Heavy-precip cutoff excluded from training baseline |
| `feature_cols` / `scale_cols` | travel, delay, weather, cyclical time, calendar flags | Model inputs; only continuous cols are scaled with **train-only** stats later |


In [ ]:
from __future__ import annotations

import json
import random
from dataclasses import asdict, dataclass
from pathlib import Path

import numpy as np
import pandas as pd

# Google Drive project root as seen from a Colab runtime (/content/drive/...).
DRIVE_ROOT = Path("/content/drive/MyDrive/Dissertation")


@dataclass(frozen=True)
class STGAEConfig:
    # --- Paths ---
    panel_path: Path = DRIVE_ROOT / "route_time_panel_v1.parquet"
    artifact_dir: Path = DRIVE_ROOT / "artifacts" / "st_gae"

    # --- Temporal train / eval protocol ---
    train_years: tuple[int, ...] = (2016,)  # Calendar years used to learn healthy (normative) traffic dynamics.
    eval_years: tuple[int, ...] = (2017,)   # Held-out year(s) for anomaly scoring and collision-based validation.

    # --- Model / training hyperparameters ---
    window_size: int = 12   # Rolling history length in 5-min steps (12 → 60 minutes of context for the GRU).
    batch_size: int = 32    # Number of window samples per optimizer step.
    hidden_dim: int = 64    # Width of GCN / GRU latent state.
    lr: float = 1e-3        # Adam learning rate for reconstruction training.
    epochs: int = 30        # Full passes over the training window dataset.
    seed: int = 42          # Seed for numpy / python RNGs (and torch later) for reproducibility.

    # --- Normative / cleaning thresholds (healthy-flow baseline) ---
    # Delay quantile computed *inside* each context stratum (see delay_context_keys) 
    # so expected rush-hour delay stays "normal" and can be learned by the ST-GAE.
    delay_percentile: float = 0.85

    # Columns that define a delay-context stratum. Keep order stable for QA/debug.
    # - route_id: corridor-specific delay scale
    # - hour: time-of-day (0–23); captures AM/PM peaks
    # - dow: day-of-week (Mon=0 … Sun=6); captures Friday vs Sunday patterns
    delay_context_keys: tuple[str, ...] = ("route_id", "hour", "dow")

    # If a stratum has fewer samples than this, fall back to coarser keys
    # (drop dow → route+hour, then route only) so sparse cells stay stable.
    delay_context_min_count: int = 30
    
    # Precipitation (mm/hour) above this is treated as non-normal weather for training.
    # Kept separate from the delay quantile so rain is an exclusion flag, not mixed into
    # the definition of "typical Rush hour delay".
    precip_threshold_mm: float = 2.5

    # Minimum Bluetooth sample_count required to treat a bin as observed.
    min_sample_count: int = 0

    # --- Feature schema ---
    # Continuous columns scaled with train-only mean/std (no eval leakage).
    scale_cols: tuple[str, ...] = (
        "travel_time_s",  # corridor travel time (seconds)
        "delay_s",  # travel_time_s - free_flow_s
        "wx_temp_c",  # hourly temperature (°C)
        "wx_precip_mm",  # hourly precipitation (mm)
    )

    # Full model input vector at each corridor × time (after cleaning / encoding).
    feature_cols: tuple[str, ...] = (
        "travel_time_s",  # scaled continuous
        "delay_s",  # scaled continuous
        "wx_temp_c",  # scaled continuous
        "wx_precip_mm",  # scaled continuous
        "hour_sin",  # cyclical hour-of-day
        "hour_cos",  # cyclical hour-of-day
        "dow_sin",  # cyclical day-of-week
        "dow_cos",  # cyclical day-of-week
        "is_public_holiday",  # calendar flag (0/1)
        "is_weekend",  # calendar flag (0/1)
    )


# Instantiate
CFG = STGAEConfig()

# Make cleaning / sampling deterministic across re-runs of this notebook.
random.seed(CFG.seed)
np.random.seed(CFG.seed)

print("DRIVE_ROOT:", DRIVE_ROOT)
print("panel_path:", CFG.panel_path)
print("artifact_dir:", CFG.artifact_dir)
print("config:", json.dumps({k: str(v) for k, v in asdict(CFG).items()}, indent=2))

# Prefer GPU when Colab provides one; fall back to CPU otherwise.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 2. Mount Google Drive and load the joined analysis panel

Mount Google Drive in Colab, then read `route_time_panel_v1.parquet` from `MyDrive/Dissertation/`.

In [ ]:
from google.colab import drive

# 1. Mount Google Drive (Colab maps it under /content/drive)
drive.mount("/content/drive")

# 2. Load dataset from the Dissertation folder on Drive
file_path = CFG.panel_path

# 3. Create artifacts directory
CFG.artifact_dir.mkdir(parents=True, exist_ok=True)

raw = pd.read_parquet(file_path)
raw["ts_local"] = pd.to_datetime(raw["ts_local"], utc=False)

print("loaded:", file_path)
print("rows:", f"{len(raw):,}")
print("years:", sorted(raw["year"].unique().tolist()))
print("routes:", raw["route_id"].nunique())
print("columns:", list(raw.columns))
display(raw.head(3))
display(raw.isna().mean().sort_values(ascending=False).head(12).to_frame("null_rate"))